# Nigeria Zero-Dose Children — Phase 1 Analytical Notebook
## NPHCDA Digital Innovation Hub | UNICEF Technical Assistance

**Principal Investigator:** Dr. Amobi Andrew Onovo, PhD, MPH  
**Consortium:** CIDRE–Quantium Insights

### Analytical components
1. Data loading, cleaning, and validation  
2. Bayesian Hierarchical Beta Regression (NDHS 2008–2024)  
3. Posterior predictive forecasts: 2026, 2027, 2028  
4. Getis-Ord Gi* spatial hotspot analysis (GRID3 boundaries)  
5. Composite risk index and state priority ranking  
6. LGA-level ranking calibrated to Bayesian state posteriors  
7. Publication-quality choropleth maps  

**Zero-dose definition (WHO/GAVI):** Children 12–23 months who have not received  
DTP1 (Pentavalent dose 1) by the end of their first year of life, even if they received  
birth-dose vaccines (BCG, OPV0, HepB0).  
*Reference: WHO IMR-7792; GAVI Phase 5 Equity Goal*


---
## Section 0 — Environment Setup
Run cells in order. Start here to detect your environment and install packages.

In [1]:
# Detect whether running in Google Colab or locally
try:
    import google.colab
    IN_COLAB = True
    print("Environment: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Environment: Local Jupyter / JupyterLab")


Environment: Google Colab


In [2]:
# Install required packages — run once per session
# In Colab this takes ~2 minutes on first run

import subprocess, sys

packages = [
    "pymc", "arviz", "geopandas", "libpysal", "esda",
    "scipy", "scikit-learn", "matplotlib", "pandas", "numpy",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("\n✅ All packages installed.")



✅ All packages installed.


### Upload Data Files
Run the three cells below to upload your CSV data files and GRID3 spatial shapefiles.

> **Tip:** Select all required files at once by holding **Ctrl** (Windows) or **Cmd** (Mac) when clicking.

In [4]:
# ─── COLAB: Upload your CSV data files ───────────────────────────────────
# Run this cell only when running in Google Colab.
# It will open a file picker — select ALL four CSV files at once
# (hold Ctrl/Cmd to select multiple files).
#
# Required files:
#   nigeria_ndhs_zero_dose_VERIFIED_long.csv
#   nigeria_zero_dose_MASTER_dataset.csv
#   dhis2_data_all_states.csv
#   under_5_2024.csv

if IN_COLAB:
    from google.colab import files
    print("📂 Select your CSV files in the file picker below...")
    uploaded = files.upload()
    for f in uploaded:
        print(f"✅ Uploaded: {f}  ({len(uploaded[f])/1024:.0f} KB)")
    print("\nAll CSV files uploaded. Proceed to the next cell.")
else:
    print("Local environment — files should already be in your working directory.")
    import os
    expected = [
        "nigeria_ndhs_zero_dose_VERIFIED_long.csv",
        "nigeria_zero_dose_MASTER_dataset.csv",
        "dhis2_data_all_states.csv",
        "under_5_2024.csv",
    ]
    for f in expected:
        status = "✅" if os.path.exists(f) else "❌ MISSING"
        print(f"  {status}  {f}")


📂 Select your CSV files in the file picker below...


Saving dhis2_data_all_states.csv to dhis2_data_all_states.csv
Saving nigeria_ndhs_zero_dose_VERIFIED_long.csv to nigeria_ndhs_zero_dose_VERIFIED_long (1).csv
Saving nigeria_zero_dose_MASTER_dataset.csv to nigeria_zero_dose_MASTER_dataset.csv
Saving under_5_2024.csv to under_5_2024.csv
✅ Uploaded: dhis2_data_all_states.csv  (5453 KB)
✅ Uploaded: nigeria_ndhs_zero_dose_VERIFIED_long (1).csv  (12 KB)
✅ Uploaded: nigeria_zero_dose_MASTER_dataset.csv  (10 KB)
✅ Uploaded: under_5_2024.csv  (1 KB)

All CSV files uploaded. Proceed to the next cell.


In [5]:
# ─── COLAB: Upload the GRID3 spatial files ───────────────────────────────
# Upload the spatial.zipx file (or spatial.zip — both are supported).
#
# Required contents inside the archive:
#   spatial/admin1/grid3_nga_boundary_vaccstates.shp  (+ .dbf .prj .shx)
#   spatial/admin2/grid3_nga_boundary_vacclgas.shp    (+ .dbf .prj .shx)

if IN_COLAB:
    from google.colab import files
    print("Select spatial.zipx (or spatial.zip) in the file picker below...")
    uploaded_zip = files.upload()
    for f in uploaded_zip:
        print(f"Uploaded: {f}  ({len(uploaded_zip[f])/1024:.0f} KB)")
else:
    print("Local environment — spatial/ folder should already exist.")
    import os
    for shp in ["spatial/admin1/grid3_nga_boundary_vaccstates.shp",
                "spatial/admin2/grid3_nga_boundary_vacclgas.shp"]:
        print(f"  {'OK' if os.path.exists(shp) else 'MISSING'}  {shp}")


Select spatial.zipx (or spatial.zip) in the file picker below...


Saving spatial.zipx to spatial.zipx
Uploaded: spatial.zipx  (6566 KB)


In [6]:
# Extract the spatial archive — supports .zip and .zipx
#
# .zipx uses WinZip extended compression not supported by Python's
# built-in zipfile module. This cell uses the system 7z tool instead,
# which handles all ZIP variants including .zipx.

import os, subprocess, sys, glob

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr

if IN_COLAB:

    # Step 1: locate the uploaded archive anywhere in /content
    archives = (glob.glob("/content/*.zipx") + glob.glob("/content/*.zip") +
                glob.glob("*.zipx") + glob.glob("*.zip"))
    archives = list(dict.fromkeys(archives))   # deduplicate

    if not archives:
        print("No archive found. Re-run Cell 06 to upload spatial.zipx")
    else:
        archive = archives[0]
        print(f"Archive located: {archive}")

        # Step 2: install p7zip-full (silent, fast no-op if already installed)
        print("Ensuring p7zip-full is installed...")
        _run("apt-get install -y p7zip-full -q 2>/dev/null")

        # Step 3: extract to /content using 7z
        print("Extracting archive...")
        extract_cmd = '7z x "' + archive + '" -o/content -y'
        rc, out = _run(extract_cmd)
        if rc != 0:
            print("7z returned an error. Trying /tmp fallback...")
            tmp_path = "/tmp/spatial_archive.zipx"
            _run('cp "' + archive + '" ' + tmp_path)
            rc2, out2 = _run("7z x " + tmp_path + " -o/content -y")
            if rc2 == 0:
                print("Extraction succeeded via /tmp fallback.")
            else:
                print("Extraction failed. Output:")
                print(out2[:400])
        else:
            print("Extraction complete.")

else:
    print("Local environment — no extraction needed.")

# Step 4: auto-discover shapefiles regardless of where they extracted to
import subprocess as _sp
_result = _sp.run("find /content -name '*.shp' 2>/dev/null",
                  shell=True, capture_output=True, text=True)
_all_shps = [p.strip() for p in _result.stdout.strip().split("\n") if p.strip()]

state_shp_found = [p for p in _all_shps if "vaccstate" in p.lower() or "admin1" in p.lower()]
lga_shp_found   = [p for p in _all_shps if "vacclga"   in p.lower() or "admin2" in p.lower()]

print("\nShapefiles discovered:")
print(f"  State : {state_shp_found[0] if state_shp_found else 'NOT FOUND'}")
print(f"  LGA   : {lga_shp_found[0]   if lga_shp_found   else 'NOT FOUND'}")

# Step 5: update PATH variables dynamically — Section 6 will use these
if state_shp_found and lga_shp_found:
    PATH_SHP_STATE = state_shp_found[0]
    PATH_SHP_LGA   = lga_shp_found[0]
    print(f"\nPATH_SHP_STATE set to: {PATH_SHP_STATE}")
    print(f"PATH_SHP_LGA   set to: {PATH_SHP_LGA}")
    print("\nBoth shapefiles ready. Proceed to Section 1.")
else:
    print("\nShapefiles not found after extraction.")
    print("Check that spatial.zipx contains:")
    print("  spatial/admin1/grid3_nga_boundary_vaccstates.shp")
    print("  spatial/admin2/grid3_nga_boundary_vacclgas.shp")
    print("Re-upload spatial.zipx in Cell 06 and re-run this cell.")


Archive located: /content/spatial.zipx
Ensuring p7zip-full is installed...
Extracting archive...
Extraction complete.

Shapefiles discovered:
  State : /content/admin1/grid3_nga_boundary_vaccstates.shp
  LGA   : /content/admin2/grid3_nga_boundary_vacclgas.shp

PATH_SHP_STATE set to: /content/admin1/grid3_nga_boundary_vaccstates.shp
PATH_SHP_LGA   set to: /content/admin2/grid3_nga_boundary_vacclgas.shp

Both shapefiles ready. Proceed to Section 1.


In [7]:
# DIAGNOSTIC — run if shapefiles show NOT FOUND in Cell 07
# Searches the entire filesystem and updates PATH variables automatically.

import subprocess, glob

result = subprocess.run("find /content -name '*.shp' 2>/dev/null",
                        shell=True, capture_output=True, text=True)
found = [p.strip() for p in result.stdout.strip().split("\n") if p.strip()]

if found:
    print(f"Found {len(found)} shapefile(s):")
    for p in found:
        print(f"  {p}")

    state_shp = [p for p in found if "vaccstate" in p.lower() or "admin1" in p.lower()]
    lga_shp   = [p for p in found if "vacclga"   in p.lower() or "admin2" in p.lower()]

    if state_shp and lga_shp:
        PATH_SHP_STATE = state_shp[0]
        PATH_SHP_LGA   = lga_shp[0]
        print(f"\nPATH_SHP_STATE set to: {PATH_SHP_STATE}")
        print(f"PATH_SHP_LGA   set to: {PATH_SHP_LGA}")
        print("\nPaths updated. Proceed to Section 1 — you do not need to")
        print("edit the configuration cell manually.")
    else:
        print("\nShapefiles found but could not identify state/LGA files.")
        print("Update PATH_SHP_STATE and PATH_SHP_LGA in Cell 10 manually.")
else:
    print("No .shp files found anywhere in /content.")
    print("The archive did not extract correctly.")
    print("Please re-upload spatial.zipx in Cell 06 and re-run Cell 07.")


Found 3 shapefile(s):
  /content/admin1/grid3_nga_boundary_vaccstates.shp
  /content/admin2/grid3_nga_boundary_vacclgas.shp
  /content/admin3/grid3_nga_boundary_vaccwards.shp

PATH_SHP_STATE set to: /content/admin1/grid3_nga_boundary_vaccstates.shp
PATH_SHP_LGA   set to: /content/admin2/grid3_nga_boundary_vacclgas.shp

Paths updated. Proceed to Section 1 — you do not need to
edit the configuration cell manually.


---
## Section 1 — Imports and Configuration
Edit file paths in the **CONFIGURATION** block below before running.

In [8]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")        # change to "inline" for Jupyter display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import pymc as pm
import arviz as az
import geopandas as gpd
from libpysal.weights import Queen, KNN
from esda.getisord import G_Local

warnings.filterwarnings("ignore")
np.random.seed(42)

# ─── CONFIGURATION — edit paths for your environment ─────────────────────
PATH_NDHS_LONG   = "nigeria_ndhs_zero_dose_VERIFIED_long.csv"
PATH_NDHS_MASTER = "nigeria_zero_dose_MASTER_dataset.csv"
PATH_DHIS2       = "dhis2_data_all_states.csv"
PATH_POPULATION  = "under_5_2024.csv"

# GRID3 administrative boundary shapefiles
# Shapefile paths — set automatically by Cell 07 if extraction succeeds.
# Only edit manually if Cell 07 could not find the files.
PATH_SHP_STATE   = "spatial/admin1/grid3_nga_boundary_vaccstates.shp"
PATH_SHP_LGA     = "spatial/admin2/grid3_nga_boundary_vacclgas.shp"

OUTPUT_DIR    = "outputs"
MCMC_DRAWS    = 3000
MCMC_TUNE     = 3000
MCMC_CHAINS   = 2
TARGET_ACCEPT = 0.92
FORECAST_YEARS = [2026, 2027, 2028]
LGA_KNN = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration ready. Output directory:", OUTPUT_DIR)


Configuration ready. Output directory: outputs


---
## Section 2 — Helper Functions and Colour Palettes

In [9]:
def normalise_name(s):
    """Standardise state/LGA name for joining across datasets."""
    return str(s).strip().title().replace("-", " ").replace("_", " ")

def clean_lga_name(s):
    """Remove LGA prefix codes and 'Local Government Area' suffix."""
    s = str(s).strip()
    s = re.sub(r"^[a-z]{2}\s+", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*local\s+government\s+area\s*$", "", s, flags=re.IGNORECASE)
    return s.strip().title()

def hotspot_class(z, p):
    """Map Gi* z-score and permutation p-value to a labelled category."""
    if   p <= 0.01 and z > 0: return "Hot Spot (p<0.01)"
    elif p <= 0.05 and z > 0: return "Hot Spot (p<0.05)"
    elif p <= 0.10 and z > 0: return "Hot Spot (p<0.10)"
    elif p <= 0.01 and z < 0: return "Cold Spot (p<0.01)"
    elif p <= 0.05 and z < 0: return "Cold Spot (p<0.05)"
    elif p <= 0.10 and z < 0: return "Cold Spot (p<0.10)"
    else:                      return "Not Significant"

def minmax_scale(x):
    """Scale series to 0–100 range."""
    r = x.max() - x.min()
    return (x - x.min()) / r * 100 if r > 0 else x * 0 + 50

HOTSPOT_COLORS = {
    "Hot Spot (p<0.01)":  "#D73027", "Hot Spot (p<0.05)":  "#FC8D59",
    "Hot Spot (p<0.10)":  "#FEE090", "Not Significant":    "#CCCCCC",
    "Cold Spot (p<0.10)": "#ABD9E9", "Cold Spot (p<0.05)": "#74ADD1",
    "Cold Spot (p<0.01)": "#4575B4",
}
TIER_COLORS = {
    "Tier 1: Critical": "#D73027", "Tier 2: High":     "#FC8D59",
    "Tier 3: Moderate": "#FEE090", "Tier 4: Lower":    "#91BFDB",
}
ZONE_COLORS = {
    "North West": "#D73027", "North East": "#FC8D59", "North Central": "#FDAE61",
    "South West": "#1A9850", "South East": "#91CF60", "South South":   "#66C2A5",
}
BG = "#F8F9FA"
print("Helper functions and colour palettes defined.")


Helper functions and colour palettes defined.


---
## Section 3 — Load and Prepare Data
Verify all four input CSV files are in the working directory before running.

In [10]:
long_df   = pd.read_csv(PATH_NDHS_LONG)
master_df = pd.read_csv(PATH_NDHS_MASTER)
dhis2_df  = pd.read_csv(PATH_DHIS2)
pop_df    = pd.read_csv(PATH_POPULATION)

print(f"NDHS longitudinal : {len(long_df)} rows — "
      f"{long_df['year'].nunique()} survey rounds x {long_df['state'].nunique()} states")
print(f"DHIS2             : {len(dhis2_df)} LGA-month rows, "
      f"{dhis2_df['lga'].nunique()} unique LGAs")
long_df.head(3)


NDHS longitudinal : 148 rows — 4 survey rounds x 37 states
DHIS2             : 43656 LGA-month rows, 774 unique LGAs


,state,zone,year,dtp1_coverage,zero_dose_pct,n_children_12_23m,source_table,source_report,data_status
0,FCT,North Central,2008,87.2,12.8,48,Table A-10.3,NDHS 2008 (FR222),CONFIRMED_FROM_PDF
1,FCT,North Central,2013,83.9,16.1,38,Table 10.3,NDHS 2013 (FR293),CONFIRMED_FROM_PDF
2,FCT,North Central,2018,85.5,14.5,41,Table 10.3,NDHS 2018 (FR359),CONFIRMED_FROM_PDF


In [11]:
# Population denominators (NPC 2024 projections)
pop_c = pop_df.iloc[1:].copy()
pop_c.columns = ["zone_abbr", "state_raw", "under5"]
pop_c["under5_n"]      = pop_c["under5"].str.replace(",","").str.strip().astype(float)
pop_c["cohort_12_23m"] = (pop_c["under5_n"] / 5).round(0)
pop_c["jk"] = (pop_c["state_raw"].str.strip().str.upper()
               .str.replace(" ","").str.replace(",ABUJA","").str.replace(",",""))
pop_lookup    = dict(zip(pop_c["jk"], pop_c["cohort_12_23m"]))
pop_u5_lookup = dict(zip(pop_c["jk"], pop_c["under5_n"]))

# DHIS2 Penta1 trend covariate: log-ratio 2024/2021 Penta1 counts (per year)
dhis2_df["penta_1_count"] = pd.to_numeric(dhis2_df["penta_1_count"], errors="coerce")
dhis2_df["period_dt"]     = pd.to_datetime(dhis2_df["period"], format="%b-%y", errors="coerce")
dhis2_df["year"]          = dhis2_df["period_dt"].dt.year
ann = dhis2_df.groupby(["state","year"])["penta_1_count"].sum().reset_index()

dhis2_trend = {}
for state in ann["state"].unique():
    s   = ann[ann["state"] == state]
    v21 = s.loc[s["year"]==2021, "penta_1_count"].sum()
    v24 = s.loc[s["year"]==2024, "penta_1_count"].sum()
    dhis2_trend[state.strip()] = (np.log(v24/v21)/3) if (v21>0 and v24>0) else 0.0

# Model index arrays
long_df["state"] = long_df["state"].str.strip()
long_df["zone"]  = long_df["zone"].str.strip()

year_mean = 2024
year_std  = long_df["year"].std()
long_df["year_std"] = (long_df["year"] - year_mean) / year_std

states       = sorted(long_df["state"].unique())
zones        = sorted(long_df["zone"].unique())
n_states     = len(states); n_zones = len(zones)
s_idx        = {s:i for i,s in enumerate(states)}
z_idx        = {z:i for i,z in enumerate(zones)}
long_df["state_idx"] = long_df["state"].map(s_idx)
long_df["zone_idx"]  = long_df["zone"].map(z_idx)

state_zone     = long_df.groupby("state")["zone"].first()
state_zone_idx = np.array([z_idx[state_zone[s]] for s in states])

eps = 1e-4
long_df["zd_prop"] = (long_df["zero_dose_pct"]/100).clip(eps, 1-eps)

dhis2_cov     = np.array([dhis2_trend.get(s,0.0) for s in states])
dhis2_cov_std = (dhis2_cov - dhis2_cov.mean()) / (dhis2_cov.std() + 1e-8)

n_obs       = long_df["n_children_12_23m"].values.astype(float)
kappa_scale = n_obs / n_obs.mean()

y_obs    = long_df["zd_prop"].values
yr_std_v = long_df["year_std"].values
st_idx_v = long_df["state_idx"].values
fc_yr_std = np.array([(y - year_mean)/year_std for y in FORECAST_YEARS])

print(f"Model arrays ready: {n_states} states | {n_zones} zones | {len(long_df)} observations")


Model arrays ready: 37 states | 6 zones | 148 observations


---
## Section 4 — Bayesian Hierarchical Beta Regression

**Model specification:**
```
Likelihood:  y_ij ~ Beta(mu * kappa, (1-mu) * kappa)
Link:        logit(mu_ij) = alpha_zone[zone(s)] + sigma_a * z_a[s]
                           + (beta_zone[zone(s)] + sigma_b * z_b[s]) * year_std
                           + gamma * DHIS2_trend[s]
Precision:   kappa_obs = kappa_base * (n_i / mean_n)
```

**Priors:** all weakly informative (Normal(0,1), HalfNormal)  
**Sampler:** NUTS — target_accept=0.92  
**Runtime:** approximately 60–90 seconds on a modern laptop

In [12]:
with pm.Model() as bayes_model:

    # Zone-level hyperpriors — partial pooling
    mu_alpha = pm.Normal("mu_alpha",   mu=0, sigma=1)
    sig_a_z  = pm.HalfNormal("sig_a_z",    sigma=0.5)
    a_z_raw  = pm.Normal("a_z_raw",    mu=0, sigma=1, shape=n_zones)
    alpha_z  = pm.Deterministic("alpha_z", mu_alpha + sig_a_z * a_z_raw)

    b_year_g = pm.Normal("b_year_g",   mu=0, sigma=1)
    sig_b_z  = pm.HalfNormal("sig_b_z",    sigma=0.3)
    b_z_raw  = pm.Normal("b_z_raw",    mu=0, sigma=1, shape=n_zones)
    beta_z   = pm.Deterministic("beta_z",  b_year_g + sig_b_z * b_z_raw)

    # State-level random effects (non-centred parameterisation)
    sig_a_s  = pm.HalfNormal("sig_a_s",    sigma=0.5)
    z_a      = pm.Normal("z_a",        mu=0, sigma=1, shape=n_states)
    alpha_s  = pm.Deterministic("alpha_s", alpha_z[state_zone_idx] + sig_a_s * z_a)

    sig_b_s  = pm.HalfNormal("sig_b_s",    sigma=0.3)
    z_b      = pm.Normal("z_b",        mu=0, sigma=1, shape=n_states)
    beta_s   = pm.Deterministic("beta_s",  beta_z[state_zone_idx] + sig_b_s * z_b)

    # DHIS2 trend covariate
    gamma    = pm.Normal("gamma",      mu=0, sigma=0.5)

    # Beta precision scales with NDHS sample size
    kappa_b  = pm.Gamma("kappa_b",    alpha=2, beta=0.5)

    # Linear predictor
    eta = (alpha_s[st_idx_v] + beta_s[st_idx_v] * yr_std_v
           + gamma * dhis2_cov_std[st_idx_v])
    mu  = pm.Deterministic("mu", pm.math.invlogit(eta))

    kappa_obs = kappa_b * kappa_scale
    _         = pm.Beta("y_like", alpha=mu*kappa_obs, beta=(1-mu)*kappa_obs,
                        observed=y_obs)

    trace = pm.sample(
        draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=MCMC_CHAINS, cores=1,
        target_accept=TARGET_ACCEPT, random_seed=42,
        progressbar=True, return_inferencedata=True,
    )
    print("Generating forecasts...")
    a_flat = trace.posterior["alpha_s"].values.reshape(-1, n_states)
    b_flat = trace.posterior["beta_s"].values.reshape(-1, n_states)
    g_flat = trace.posterior["gamma"].values.reshape(-1)

    fc_mu  = np.zeros((a_flat.shape[0], n_states, len(FORECAST_YEARS)))
    for fi, fys in enumerate(fc_yr_std):
        eta_fc = a_flat + b_flat*fys + g_flat[:,None]*dhis2_cov_std[None,:]
        fc_mu[:,:,fi] = 1 / (1 + np.exp(-eta_fc))

print("Sampling complete.")


Output()

Generating forecasts...
Sampling complete.


### 4.1 Convergence Diagnostics
Confirm Max R-hat < 1.01 and Min ESS > 400 before proceeding.

In [13]:
rhat_v  = az.rhat(trace)
max_rh  = float(max(rhat_v[v].values.max() for v in rhat_v.data_vars))
ess_v   = az.ess(trace, method="bulk")
min_ess = float(min(ess_v[v].values.min() for v in ess_v.data_vars))

print(f"Max R-hat : {max_rh:.4f}  ({'PASS' if max_rh < 1.01 else 'FAIL — re-run with more tuning steps'})")
print(f"Min ESS   : {min_ess:.0f}  ({'PASS' if min_ess > 400 else 'WARNING — increase MCMC_DRAWS'})")

# Save diagnostics
az.summary(trace, var_names=["mu_alpha","b_year_g","sig_a_s","sig_b_s","gamma","kappa_b"])  .to_csv(f"{OUTPUT_DIR}/model_diagnostics.csv")
print(f"Diagnostics saved to {OUTPUT_DIR}/model_diagnostics.csv")


Max R-hat : 1.0017  (PASS)
Min ESS   : 2171  (PASS)
Diagnostics saved to outputs/model_diagnostics.csv


### 4.2 Posterior Predictive Check

In [14]:
# Trace plot — verify chain mixing
az.plot_trace(trace, var_names=["mu_alpha","b_year_g","gamma","kappa_b"],
              figsize=(14, 6), compact=True)
plt.suptitle("Trace and Density Plots — Key Parameters", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


---
## Section 5 — Extract Posterior Results and Build State Table

In [15]:
rows = []
for si, state in enumerate(states):
    sdf  = long_df[long_df["state"] == state]
    zone = state_zone[state]
    jk   = state.upper().replace(" ","")
    pop  = pop_lookup.get(jk, np.nan)

    def obs(yr):
        v = sdf.loc[sdf["year"]==yr, "zero_dose_pct"].values
        return float(v[0]) if len(v) else np.nan

    r = dict(state=state, zone=zone, pop_under5=pop_u5_lookup.get(jk,np.nan),
             cohort_12_23m=pop, state_key=normalise_name(state),
             zd_obs_2008=obs(2008), zd_obs_2013=obs(2013),
             zd_obs_2018=obs(2018), zd_obs_2024=obs(2024))

    for fi, yr in enumerate(FORECAST_YEARS):
        draws = fc_mu[:, si, fi] * 100
        r[f"zd_pred_{yr}_mean"] = float(np.mean(draws))
        r[f"zd_pred_{yr}_lo95"] = float(np.percentile(draws, 2.5))
        r[f"zd_pred_{yr}_hi95"] = float(np.percentile(draws, 97.5))
        r[f"zd_count_{yr}"]     = float(np.mean(draws))/100 * pop if not np.isnan(pop) else np.nan
    rows.append(r)

res = pd.DataFrame(rows)

# Composite risk index
res["score_rate"]   = minmax_scale(res["zd_pred_2026_mean"])
res["score_count"]  = minmax_scale(res["zd_count_2026"])
res["score_trend"]  = minmax_scale(res["zd_obs_2024"] - res["zd_obs_2018"])
res["score_uncert"] = minmax_scale(res["zd_pred_2026_hi95"] - res["zd_pred_2026_lo95"])
res["risk_index"]   = (0.45*res["score_rate"] + 0.30*res["score_count"]
                       + 0.15*res["score_trend"] + 0.10*res["score_uncert"])
res["state_rank"]   = res["risk_index"].rank(ascending=False).astype(int)
res["priority_tier"] = pd.cut(res["risk_index"],
    bins=[-np.inf,25,50,75,np.inf],
    labels=["Tier 4: Lower","Tier 3: Moderate","Tier 2: High","Tier 1: Critical"])
res = res.sort_values("state_rank").reset_index(drop=True)

print("Top 10 highest-risk states:")
res[["state_rank","state","zone","zd_obs_2024","zd_pred_2026_mean",
     "zd_count_2026","risk_index","priority_tier"]].head(10).round(1)


Top 10 highest-risk states:


,state_rank,state,zone,zd_obs_2024,zd_pred_2026_mean,zd_count_2026,risk_index,priority_tier
0,1,Sokoto,North West,86.5,71.9,168836.6,85.2,Tier 1: Critical
1,2,Kebbi,North West,84.0,68.3,154105.4,81.7,Tier 1: Critical
2,3,Zamfara,North West,82.6,73.1,156169.0,81.0,Tier 1: Critical
3,4,Kano,North West,42.4,36.4,194931.8,61.1,Tier 2: High
4,5,Kaduna,North West,45.6,43.4,138382.8,58.4,Tier 2: High
5,6,Niger,North Central,56.0,45.2,104497.7,57.5,Tier 2: High
6,7,Katsina,North West,39.8,38.1,165813.2,54.9,Tier 2: High
7,8,Kogi,North Central,57.0,32.2,48946.4,48.6,Tier 3: Moderate
8,9,Kwara,North Central,51.7,37.4,47027.9,46.5,Tier 3: Moderate
9,10,Jigawa,North West,34.3,33.9,103403.4,45.6,Tier 3: Moderate


In [16]:
# Generate LGA zero-dose rankings from DHIS2 + Bayesian state predictions
# Runs automatically after state results table — no upload needed.

import os

_lga_out = f"{OUTPUT_DIR}/lga_zero_dose_ranked.csv"

if os.path.exists(_lga_out):
    print(f"lga_zero_dose_ranked.csv already exists — skipping generation.")
else:
    print("Generating LGA rankings from DHIS2 + Bayesian state predictions...")

    # Ensure numeric columns are parsed
    for _col in ["penta_1_count","penta_3_count","pent_2_count","measles_1_count"]:
        if _col in dhis2_df.columns:
            dhis2_df[_col] = pd.to_numeric(dhis2_df[_col], errors="coerce")

    # Ensure year column exists
    if "year" not in dhis2_df.columns:
        dhis2_df["period_dt"] = pd.to_datetime(
            dhis2_df["period"], format="%b-%y", errors="coerce")
        dhis2_df["year"] = dhis2_df["period_dt"].dt.year

    # Determine which zone/group columns exist
    _grp_cols = ["state","lga"]
    if "zone" in dhis2_df.columns:
        _grp_cols = ["zone","state","lga"]

    # Aggregate to LGA level — use most recent year with non-zero Penta1
    _best_rows = []
    for _yr in [2024, 2023, 2022, 2021]:
        _yr_df = (dhis2_df[dhis2_df["year"] == _yr]
                  .groupby(_grp_cols)[["penta_1_count","penta_3_count"]]
                  .sum().reset_index())
        _yr_df["yr"] = _yr
        _best_rows.append(_yr_df)

    _lga_all  = pd.concat(_best_rows, ignore_index=True)
    _lga_all  = _lga_all[_lga_all["penta_1_count"] > 0]
    _lga_best = (_lga_all.sort_values(["state","lga","yr"],
                                       ascending=[True,True,False])
                 .drop_duplicates(subset=["state","lga"], keep="first")
                 .reset_index(drop=True))

    print(f"  DHIS2 LGAs available: {len(_lga_best)}")

    # State-level Bayesian predictions — use state name directly (title case)
    _state_pred = dict(zip(res["state"], res["zd_pred_2026_mean"] / 100))
    # Population lookup — try both title-case name and JK key
    _state_pop_by_name = dict(zip(res["state"], res["cohort_12_23m"]))

    _out_rows = []
    _skipped  = 0
    for _state, _sg in _lga_best.groupby("state"):
        _state_clean = str(_state).strip()

        # Try exact match first, then normalised
        _zd = (_state_pred.get(_state_clean) or
               _state_pred.get(_state_clean.title()) or
               _state_pred.get(normalise_name(_state_clean)))
        _pop = (_state_pop_by_name.get(_state_clean) or
                _state_pop_by_name.get(_state_clean.title()) or
                _state_pop_by_name.get(normalise_name(_state_clean)))

        if _zd is None or (isinstance(_zd, float) and np.isnan(_zd)):
            _skipped += 1
            continue

        _sg  = _sg.copy()
        _tot = _sg["penta_1_count"].sum()
        if _tot == 0:
            continue

        _sg["p1_share"] = _sg["penta_1_count"] / _tot
        _sg["dropout_p1p3"] = (
            (_sg["penta_1_count"] - _sg["penta_3_count"])
            / _sg["penta_1_count"].replace(0, np.nan) * 100
        ).clip(-50, 100)

        # Calibrate LGA proxy rate to state Bayesian posterior mean
        _mean_share = _sg["p1_share"].mean()
        _sg["zd_proxy_raw"] = _zd * (
            1 + (_mean_share - _sg["p1_share"]) / (_mean_share + 1e-6)
        )
        _sg["zd_proxy_pct"] = (_sg["zd_proxy_raw"] * 100).clip(1, 99)

        if _pop is not None and not (isinstance(_pop, float) and np.isnan(_pop)):
            _sg["lga_pop_est"]  = float(_pop) / len(_sg)
            _sg["zd_count_est"] = (
                _sg["zd_proxy_pct"] / 100 * _sg["lga_pop_est"]
            ).round(0)
        else:
            _sg["lga_pop_est"]  = np.nan
            _sg["zd_count_est"] = np.nan

        _out_rows.append(_sg)

    if not _out_rows:
        print("No LGA rows generated — check state name matching between DHIS2 and model dataset.")
    else:
        _lga_df = pd.concat(_out_rows, ignore_index=True)

        def _mm(x):
            r = x.max() - x.min()
            return (x - x.min()) / r * 100 if r > 0 else x * 0 + 50

        _lga_df["score_rate"] = _lga_df.groupby("state")["zd_proxy_pct"].transform(_mm)
        _lga_df["score_dropout"] = _lga_df.groupby("state")["dropout_p1p3"].transform(
            lambda x: _mm(x.fillna(x.median())))
        _lga_df["lga_risk_index"] = (0.60 * _lga_df["score_rate"]
                                     + 0.40 * _lga_df["score_dropout"])
        _lga_df["national_lga_rank"] = _lga_df["lga_risk_index"].rank(
            ascending=False, method="min").astype(int)
        _lga_df["state_lga_rank"] = _lga_df.groupby("state")["lga_risk_index"].rank(
            ascending=False, method="min").astype(int)
        _lga_df["lga_tier"] = pd.cut(
            _lga_df["lga_risk_index"],
            bins=[-np.inf, 25, 50, 75, np.inf],
            labels=["Tier 4","Tier 3","Tier 2","Tier 1"]
        )

        _save_cols = ["national_lga_rank","state_lga_rank","zone","state","lga",
                      "penta_1_count","penta_3_count","p1_share","zd_proxy_pct",
                      "zd_count_est","dropout_p1p3","lga_risk_index","lga_tier","yr"]
        _save_cols = [c for c in _save_cols if c in _lga_df.columns]
        _lga_df[_save_cols].to_csv(_lga_out, index=False)

        print(f"Generated: {_lga_out}")
        print(f"  LGAs            : {len(_lga_df)}")
        print(f"  States covered  : {_lga_df['state'].nunique()} "
              f"(skipped {_skipped} with no Bayesian match)")
        print(f"  Tier 1 LGAs     : {(_lga_df['lga_tier']=='Tier 1').sum()}")
        print(f"  Mean ZD proxy   : {_lga_df['zd_proxy_pct'].mean():.1f}%")
        print(f"  ZD proxy range  : {_lga_df['zd_proxy_pct'].min():.1f}% – "
              f"{_lga_df['zd_proxy_pct'].max():.1f}%")


Generating LGA rankings from DHIS2 + Bayesian state predictions...
  DHIS2 LGAs available: 730
Generated: outputs/lga_zero_dose_ranked.csv
  LGAs            : 730
  States covered  : 37 (skipped 0 with no Bayesian match)
  Tier 1 LGAs     : 103
  Mean ZD proxy   : 26.5%
  ZD proxy range  : 1.0% – 99.0%


---
## Section 6 — Getis-Ord Gi* Spatial Hotspot Analysis (GRID3 Boundaries)

**State level:** Queen contiguity weights from GRID3 Admin1 shapefiles  
**LGA level:** k-nearest neighbours (k=5) from GRID3 Admin2 shapefiles  
**Inference:** 999 conditional randomisation permutations  

The Gi* statistic tests whether high (or low) zero-dose rates are spatially clustered beyond what would be expected by chance. A positive z-score with p≤0.05 indicates a statistically significant hot spot.

In [17]:
# Load GRID3 shapefiles
# This cell auto-discovers shapefile paths if the defaults are wrong.
# It does not require re-uploading or re-extracting — it searches first.

import os, subprocess

def _find_shp(keyword):
    """Search /content for a shapefile matching keyword."""
    result = subprocess.run(
        f"find /content -name '*.shp' 2>/dev/null",
        shell=True, capture_output=True, text=True
    )
    matches = [p.strip() for p in result.stdout.strip().split("\n")
               if p.strip() and keyword.lower() in p.lower()]
    return matches[0] if matches else None

# Resolve paths: use configured value if it exists, otherwise auto-discover
_state_path = PATH_SHP_STATE if os.path.exists(PATH_SHP_STATE) else _find_shp("vaccstate") or _find_shp("admin1")
_lga_path   = PATH_SHP_LGA   if os.path.exists(PATH_SHP_LGA)   else _find_shp("vacclga")   or _find_shp("admin2")

if _state_path is None or _lga_path is None:
    print("Shapefiles could not be located. Checklist:")
    print("  1. Did Cell 07 print 'Extraction complete'?")
    print("     If not, re-run Cell 06 (upload) and Cell 07 (extract).")
    print("  2. Run Cell 08 (diagnostic) to search for .shp files.")
    print("  3. If files are found at unexpected paths, update PATH_SHP_STATE")
    print("     and PATH_SHP_LGA in Cell 10 then re-run this cell.")
    raise FileNotFoundError(
        "GRID3 shapefiles not found. See instructions printed above."
    )

# Update module-level path variables so figure cells use correct paths
PATH_SHP_STATE = _state_path
PATH_SHP_LGA   = _lga_path
print(f"State shapefile : {PATH_SHP_STATE}")
print(f"LGA shapefile   : {PATH_SHP_LGA}")

# Load shapefiles
gdf_st  = gpd.read_file(PATH_SHP_STATE)
gdf_lga = gpd.read_file(PATH_SHP_LGA)

gdf_st["state_key"] = gdf_st["statename"].apply(normalise_name)
geo = gdf_st.merge(res, on="state_key", how="left").to_crs(epsg=4326)

# Queen contiguity weights (shared borders) — UTM 32N for Nigeria
geo_proj = geo.to_crs(epsg=32632)
w_q      = Queen.from_dataframe(geo_proj)
w_q.transform = "r"

gi_cols = []
for yr in FORECAST_YEARS:
    y   = geo[f"zd_pred_{yr}_mean"].fillna(geo[f"zd_pred_{yr}_mean"].mean()).values
    gi  = G_Local(y, w_q, star=True, transform="r", permutations=999, seed=42)
    geo[f"gi_z_{yr}"]     = gi.Zs
    geo[f"gi_p_{yr}"]     = gi.p_sim
    geo[f"gi_class_{yr}"] = [hotspot_class(z, p) for z, p in zip(gi.Zs, gi.p_sim)]
    gi_cols += [f"gi_z_{yr}", f"gi_p_{yr}", f"gi_class_{yr}"]

res = res.merge(geo[["state_key"] + gi_cols], on="state_key", how="left")

print("\n2026 Gi* results (Queen contiguity, GRID3):")
res[["state", "zd_pred_2026_mean", "gi_z_2026", "gi_class_2026"]] \
   .sort_values("gi_z_2026", ascending=False)


State shapefile : /content/admin1/grid3_nga_boundary_vaccstates.shp
LGA shapefile   : /content/admin2/grid3_nga_boundary_vacclgas.shp

2026 Gi* results (Queen contiguity, GRID3):


,state,zd_pred_2026_mean,gi_z_2026,gi_class_2026
0,Sokoto,71.902291,2.594741,Hot Spot (p<0.01)
1,Kebbi,68.270106,2.220278,Hot Spot (p<0.01)
2,Zamfara,73.059812,1.760564,Hot Spot (p<0.01)
5,Niger,45.156568,1.100343,Hot Spot (p<0.01)
6,Katsina,38.096517,1.085314,Hot Spot (p<0.05)
4,Kaduna,43.364699,0.667436,Hot Spot (p<0.05)
3,Kano,36.387900,0.637223,Not Significant
9,Jigawa,33.872107,0.541210,Not Significant
10,Bauchi,34.287952,0.444257,Not Significant
13,Yobe,35.056976,0.430150,Not Significant


In [18]:
# LGA-level Gi* using k=5 nearest neighbours (GRID3 Admin2 boundaries)
# This cell is self-contained: it generates lga_zero_dose_ranked.csv
# if it does not already exist, then runs the Gi* analysis.
# It will always work even if Cell 24 was skipped or errored.

import os

_lga_csv = f"{OUTPUT_DIR}/lga_zero_dose_ranked.csv"

# ── Generate if missing ────────────────────────────────────────────
if not os.path.exists(_lga_csv):
    print("lga_zero_dose_ranked.csv not found — generating now from DHIS2 data...")

    for _col in ["penta_1_count","penta_3_count","pent_2_count","measles_1_count"]:
        if _col in dhis2_df.columns:
            dhis2_df[_col] = pd.to_numeric(dhis2_df[_col], errors="coerce")

    if "year" not in dhis2_df.columns:
        dhis2_df["period_dt"] = pd.to_datetime(
            dhis2_df["period"], format="%b-%y", errors="coerce")
        dhis2_df["year"] = dhis2_df["period_dt"].dt.year

    _grp = ["state","lga"] + (["zone"] if "zone" in dhis2_df.columns else [])

    _best = []
    for _yr in [2024, 2023, 2022, 2021]:
        _d = (dhis2_df[dhis2_df["year"]==_yr]
              .groupby(_grp)[["penta_1_count","penta_3_count"]]
              .sum().reset_index())
        _d["yr"] = _yr
        _best.append(_d)

    _all = pd.concat(_best, ignore_index=True)
    _all = _all[_all["penta_1_count"] > 0]
    _best_df = (_all.sort_values(["state","lga","yr"], ascending=[True,True,False])
                .drop_duplicates(subset=["state","lga"], keep="first")
                .reset_index(drop=True))

    _sp = dict(zip(res["state"], res["zd_pred_2026_mean"] / 100))
    _pp = dict(zip(res["state"], res["cohort_12_23m"]))

    _rows = []
    for _st, _sg in _best_df.groupby("state"):
        _sc = str(_st).strip()
        _zd  = _sp.get(_sc) or _sp.get(_sc.title()) or _sp.get(normalise_name(_sc))
        _pop = _pp.get(_sc) or _pp.get(_sc.title()) or _pp.get(normalise_name(_sc))
        if _zd is None or np.isnan(float(_zd)):
            continue
        _sg = _sg.copy()
        _tot = _sg["penta_1_count"].sum()
        if _tot == 0:
            continue
        _sg["p1_share"] = _sg["penta_1_count"] / _tot
        _sg["dropout_p1p3"] = (
            (_sg["penta_1_count"] - _sg["penta_3_count"])
            / _sg["penta_1_count"].replace(0, np.nan) * 100).clip(-50, 100)
        _ms = _sg["p1_share"].mean()
        _sg["zd_proxy_pct"] = (
            float(_zd) * (1 + (_ms - _sg["p1_share"]) / (_ms + 1e-6)) * 100
        ).clip(1, 99)
        if _pop and not np.isnan(float(_pop)):
            _sg["lga_pop_est"]  = float(_pop) / len(_sg)
            _sg["zd_count_est"] = (_sg["zd_proxy_pct"]/100 * _sg["lga_pop_est"]).round(0)
        else:
            _sg["lga_pop_est"] = np.nan; _sg["zd_count_est"] = np.nan
        _rows.append(_sg)

    _lga_df = pd.concat(_rows, ignore_index=True)

    def _mm(x):
        r = x.max()-x.min(); return (x-x.min())/r*100 if r>0 else x*0+50

    _lga_df["score_rate"]     = _lga_df.groupby("state")["zd_proxy_pct"].transform(_mm)
    _lga_df["score_dropout"]  = _lga_df.groupby("state")["dropout_p1p3"].transform(
        lambda x: _mm(x.fillna(x.median())))
    _lga_df["lga_risk_index"] = 0.60*_lga_df["score_rate"] + 0.40*_lga_df["score_dropout"]
    _lga_df["national_lga_rank"] = _lga_df["lga_risk_index"].rank(
        ascending=False, method="min").astype(int)
    _lga_df["state_lga_rank"] = _lga_df.groupby("state")["lga_risk_index"].rank(
        ascending=False, method="min").astype(int)
    _lga_df["lga_tier"] = pd.cut(_lga_df["lga_risk_index"],
        bins=[-np.inf,25,50,75,np.inf], labels=["Tier 4","Tier 3","Tier 2","Tier 1"])

    _sc = [c for c in ["national_lga_rank","state_lga_rank","zone","state","lga",
           "penta_1_count","penta_3_count","p1_share","zd_proxy_pct","zd_count_est",
           "dropout_p1p3","lga_risk_index","lga_tier","yr"] if c in _lga_df.columns]
    _lga_df[_sc].to_csv(_lga_csv, index=False)
    print(f"Generated: {len(_lga_df)} LGAs across {_lga_df['state'].nunique()} states")
else:
    print(f"Using existing lga_zero_dose_ranked.csv")

# ── Load and run Gi* ───────────────────────────────────────────────
lga_df = pd.read_csv(_lga_csv)
lga_df["state_key"] = lga_df["state"].apply(normalise_name)
lga_df["lga_key"]   = lga_df["lga"].apply(clean_lga_name)

gdf_lga["state_key"] = gdf_lga["statename"].apply(normalise_name)
gdf_lga["lga_key"]   = gdf_lga["lganame"].apply(clean_lga_name)

geo_lga = gdf_lga.merge(
    lga_df[["state_key","lga_key","zd_proxy_pct",
            "lga_risk_index","lga_tier","dropout_p1p3"]],
    on=["state_key","lga_key"], how="left")
geo_lga["zd_proxy_pct"] = geo_lga["zd_proxy_pct"].fillna(
    geo_lga["zd_proxy_pct"].mean())

w_lga = KNN.from_dataframe(geo_lga.to_crs(epsg=32632), k=LGA_KNN)
w_lga.transform = "r"
y_lga = geo_lga["zd_proxy_pct"].fillna(geo_lga["zd_proxy_pct"].mean()).values
gi_l  = G_Local(y_lga, w_lga, star=True, transform="r", permutations=999, seed=42)

geo_lga["gi_z_lga"]     = gi_l.Zs
geo_lga["gi_p_lga"]     = gi_l.p_sim
geo_lga["gi_class_lga"] = [hotspot_class(z,p) for z,p in zip(gi_l.Zs, gi_l.p_sim)]

n_hs = (geo_lga["gi_class_lga"]=="Hot Spot (p<0.01)").sum()
print(f"LGA Gi* complete: {n_hs} hot spots (p<0.01) out of {len(geo_lga)} LGAs")

display(
    geo_lga[["statename","lganame","gi_z_lga","gi_p_lga","gi_class_lga","zd_proxy_pct"]]
    .sort_values("gi_z_lga", ascending=False).head(15).reset_index(drop=True))


Using existing lga_zero_dose_ranked.csv
LGA Gi* complete: 55 hot spots (p<0.01) out of 774 LGAs


,statename,lganame,gi_z_lga,gi_p_lga,gi_class_lga,zd_proxy_pct
0,Sokoto,Goronyo,3.182481,0.001,Hot Spot (p<0.01),99.000000
1,Sokoto,Gada,2.840721,0.001,Hot Spot (p<0.01),99.000000
2,Zamfara,Kaura Namoda,2.831427,0.001,Hot Spot (p<0.01),99.000000
3,Zamfara,Gusau,2.483670,0.001,Hot Spot (p<0.01),99.000000
4,Sokoto,Wurno,2.342247,0.001,Hot Spot (p<0.01),98.779598
5,Zamfara,Tsafe,2.329652,0.001,Hot Spot (p<0.01),84.913534
6,Zamfara,Shinkafi,2.280700,0.001,Hot Spot (p<0.01),1.000000
7,Sokoto,Isa,2.280700,0.001,Hot Spot (p<0.01),64.541599
8,Kebbi,Augie,2.266434,0.001,Hot Spot (p<0.01),57.307132
9,Sokoto,Sabon Birni,2.225782,0.001,Hot Spot (p<0.01),89.152423


---
## Section 7 — Figures
All maps use GRID3 administrative boundaries. Change `matplotlib.use('Agg')` to `'inline'` at the top of the notebook if figures are not displaying in your environment.

In [19]:
# Figure 1 — State forecast trajectories
obs_yrs = [2008, 2013, 2018, 2024]
fig, ax  = plt.subplots(figsize=(15, 10), facecolor=BG)
ax.set_facecolor(BG)

for _, r in res.head(20).iterrows():
    c  = ZONE_COLORS.get(r["zone"], "#888")
    ov = [r[f"zd_obs_{y}"] for y in obs_yrs]
    fm = [r["zd_obs_2024"]] + [r[f"zd_pred_{y}_mean"] for y in FORECAST_YEARS]
    fl = [r["zd_obs_2024"]] + [r[f"zd_pred_{y}_lo95"] for y in FORECAST_YEARS]
    fh = [r["zd_obs_2024"]] + [r[f"zd_pred_{y}_hi95"] for y in FORECAST_YEARS]
    fy = [2024] + FORECAST_YEARS
    ax.plot(obs_yrs, ov, "o-", color=c, lw=1.8, ms=4.5, alpha=0.9)
    ax.plot(fy, fm, "--", color=c, lw=1.8, alpha=0.9)
    ax.fill_between(fy, fl, fh, color=c, alpha=0.08)
    ax.annotate(r["state"], (2028.15, r["zd_pred_2028_mean"]),
                fontsize=7.5, va="center", color=c, fontweight="bold")

ax.axvline(2024, color="#444", ls=":", lw=1.2, alpha=0.7)
ax.text(2024.2, 97, "Forecast →", fontsize=9.5, style="italic", color="#444")
ax.set_xticks([2008,2013,2018,2024,2026,2027,2028])
ax.set_xticklabels(["2008","2013","2018","2024","2026","2027","2028"], fontsize=11)
ax.set_xlim(2006, 2030); ax.set_ylim(0, 102)
ax.set_xlabel("Survey Year", fontsize=13)
ax.set_ylabel("DTP1 Non-Coverage — Zero-Dose Rate (%)", fontsize=13)
ax.set_title("Nigeria State-Level Zero-Dose Trajectories and Forecasts (2008–2028)\n"
             "Bayesian Hierarchical Beta Regression | Shaded area: 95% Credible Intervals",
             fontsize=12, fontweight="bold", pad=12)
zp = [mpatches.Patch(color=v,label=k) for k,v in ZONE_COLORS.items()]
ep = [Line2D([0],[0],color="#555",lw=1.8,marker="o",ms=5,label="Observed (NDHS)"),
      Line2D([0],[0],color="#555",lw=1.8,ls="--",label="Forecast"),
      mpatches.Patch(color="#BBBBBB",alpha=0.5,label="95% Credible Interval")]
ax.legend(handles=zp+ep, loc="upper right", fontsize=8.5, ncol=2,
          framealpha=0.92, title="Zone", title_fontsize=9)
ax.grid(alpha=0.15, ls="--")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig01_state_trajectories.png", dpi=160, bbox_inches="tight")
plt.show()
print("Figure saved.")


Figure saved.


In [20]:
# Figure 2 — Forest plot: predicted ZD rate by state with 95% credible intervals
# Sorted descending by predicted rate, coloured by priority tier, black diamonds
# overlay the observed 2024 NDHS value for visual calibration.

_df = res.sort_values("zd_pred_2026_mean", ascending=True).reset_index(drop=True)
y_pos = np.arange(len(_df))

fig, ax = plt.subplots(figsize=(12, 14), facecolor=BG)
ax.set_facecolor(BG)

bar_colors = [TIER_COLORS.get(str(t), "#888") for t in _df["priority_tier"]]
ax.barh(y_pos, _df["zd_pred_2026_mean"], color=bar_colors,
        edgecolor="white", linewidth=0.5, alpha=0.9, height=0.7)

# 95% credible intervals
err_lo = _df["zd_pred_2026_mean"] - _df["zd_pred_2026_lo95"]
err_hi = _df["zd_pred_2026_hi95"] - _df["zd_pred_2026_mean"]
ax.errorbar(_df["zd_pred_2026_mean"], y_pos,
            xerr=[err_lo, err_hi], fmt="none", ecolor="#222",
            elinewidth=1.0, capsize=3, capthick=1.0, alpha=0.85)

# Observed 2024 NDHS markers
ax.scatter(_df["zd_obs_2024"], y_pos, marker="D", s=42,
           color="black", edgecolor="white", linewidth=0.6, zorder=10)

ax.set_yticks(y_pos)
ax.set_yticklabels(_df["state"], fontsize=10)
ax.set_xlabel("DTP1 Non-Coverage — Zero-Dose Rate (%)", fontsize=12)
ax.set_xlim(0, 100)
ax.set_ylim(-0.7, len(_df) - 0.3)
ax.set_title("Predicted Zero-Dose Rate by State with 95% Credible Intervals (2026)\n"
             "Bayesian Hierarchical Beta Regression | Nigeria | NDHS 2008–2024",
             fontsize=12.5, fontweight="bold", pad=14)
ax.grid(axis="x", alpha=0.2, ls="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

tier_handles = [mpatches.Patch(color=v, label=k) for k, v in TIER_COLORS.items()]
obs_handle   = Line2D([0], [0], marker="D", color="w", markerfacecolor="black",
                      markeredgecolor="white", markersize=8, label="Observed 2024 (NDHS)")
ax.legend(handles=tier_handles + [obs_handle], loc="lower right",
          fontsize=9.5, framealpha=0.95, edgecolor="#CCC")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig02_state_forest_2026.png",
            dpi=160, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved: fig02_state_forest_2026.png")


Saved: fig02_state_forest_2026.png


In [21]:
# Figure 3 — Gi* hotspot choropleth, 3-year panel (2026, 2027, 2028) on GRID3 boundaries
fig, axes = plt.subplots(1, 3, figsize=(21, 8), facecolor=BG)

for ax, yr in zip(axes, FORECAST_YEARS):
    geo["_c"] = geo[f"gi_class_{yr}"].map(HOTSPOT_COLORS).fillna("#CCC")
    geo.plot(ax=ax, color=geo["_c"], edgecolor="white", lw=0.8, alpha=0.95)
    gdf_lga.plot(ax=ax, facecolor="none", edgecolor="#999", lw=0.18, alpha=0.35)
    for _, rr in geo.iterrows():
        c = rr.geometry.centroid
        if rr[f"zd_pred_{yr}_mean"] > 28:
            ax.annotate(rr["state_key"], (c.x, c.y), ha="center", va="center",
                        fontsize=5.5, color="#111",
                        fontweight="bold" if "Hot" in str(rr[f"gi_class_{yr}"]) else "normal")
    ax.set_title(f"Zero-Dose Child Hotspot Clusters — {yr}\n"
                 f"Getis-Ord Gi* | Queen Contiguity | GRID3 Boundaries",
                 fontsize=10.5, fontweight="bold", pad=8)
    ax.set_xlabel("Longitude", fontsize=8.5)
    ax.set_ylabel("Latitude" if yr==FORECAST_YEARS[0] else "", fontsize=8.5)
    ax.set_facecolor("#E8F4F8")

hs_h = [mpatches.Patch(color=v,label=k) for k,v in HOTSPOT_COLORS.items()]
fig.legend(handles=hs_h, loc="lower center", ncol=4, fontsize=9,
           title="Getis-Ord Gi* Classification", title_fontsize=10,
           bbox_to_anchor=(0.5,-0.04), framealpha=0.95)
fig.suptitle("Nigeria — Spatial Evolution of Zero-Dose Child Hotspots (2026–2028)\n"
             "Getis-Ord Gi* | Bayesian Posterior Predicted Means | GRID3 Admin Boundaries",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig03_gi_hotspot_maps_3yr.png", dpi=160, bbox_inches="tight")
plt.show()


In [22]:
# Figure 4 — Absolute zero-dose burden by state (2026)
# Sorted descending by estimated absolute count of zero-dose children. Bars are
# coloured by priority tier so high-rate-low-population states do not visually
# outweigh high-count states (e.g. Kano vs Zamfara — see methodology notes).

_df = res.sort_values("zd_count_2026", ascending=True).reset_index(drop=True)
y_pos = np.arange(len(_df))

fig, ax = plt.subplots(figsize=(11, 13), facecolor=BG)
ax.set_facecolor(BG)

bar_colors = [TIER_COLORS.get(str(t), "#888") for t in _df["priority_tier"]]
vals_k = _df["zd_count_2026"] / 1000.0

ax.barh(y_pos, vals_k, color=bar_colors,
        edgecolor="white", linewidth=0.5, alpha=0.9, height=0.7)

# Value labels at end of bars ("Xk")
_xmax = float(np.nanmax(vals_k)) if np.isfinite(np.nanmax(vals_k)) else 0
for i, v in enumerate(vals_k):
    if pd.notna(v):
        ax.text(v + _xmax * 0.008, i, f"{v:.0f}k",
                va="center", fontsize=9, color="#222")

ax.set_yticks(y_pos)
ax.set_yticklabels(_df["state"], fontsize=10)
ax.set_xlabel("Predicted Zero-Dose Children (thousands)", fontsize=12)
ax.set_xlim(0, _xmax * 1.10 if _xmax > 0 else 1)
ax.set_ylim(-0.7, len(_df) - 0.3)
ax.set_title("Predicted Absolute Zero-Dose Burden by State — 2026\n"
             "Estimated Children 12–23 Months | DTP1 Non-Coverage | Nigeria",
             fontsize=12.5, fontweight="bold", pad=14)
ax.grid(axis="x", alpha=0.2, ls="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

tier_handles = [mpatches.Patch(color=v, label=k) for k, v in TIER_COLORS.items()]
ax.legend(handles=tier_handles, loc="lower right",
          fontsize=9.5, framealpha=0.95, edgecolor="#CCC")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig04_absolute_burden_2026.png",
            dpi=160, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved: fig04_absolute_burden_2026.png")


Saved: fig04_absolute_burden_2026.png


In [23]:
# Figure 5 — LGA-level Gi* choropleth (774 LGAs, GRID3 Admin2 boundaries)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9), facecolor=BG)

geo_lga["_c"] = geo_lga["gi_class_lga"].map(HOTSPOT_COLORS).fillna("#CCC")
geo_lga.plot(ax=ax1, color=geo_lga["_c"], edgecolor="white", lw=0.25, alpha=0.9)
gdf_st.to_crs(epsg=4326).plot(ax=ax1, facecolor="none", edgecolor="black", lw=1.1, alpha=0.8)
ax1.legend(handles=[mpatches.Patch(color=v,label=k) for k,v in HOTSPOT_COLORS.items()
                    if k in geo_lga["gi_class_lga"].values],
           loc="lower left", fontsize=7.5, title="Gi* Classification", title_fontsize=8.5)
ax1.set_title("LGA-Level Zero-Dose Hotspot Clusters\n"
              "Getis-Ord Gi* (k=5 Nearest Neighbours) | 774 LGAs",
              fontsize=11, fontweight="bold")
ax1.set_xlabel("Longitude"); ax1.set_ylabel("Latitude"); ax1.set_facecolor("#E8F4F8")

geo_lga.plot(ax=ax2, column="zd_proxy_pct", cmap="RdYlGn_r", vmin=0, vmax=99,
             edgecolor="white", lw=0.25, alpha=0.9, missing_kwds={"color":"#DDD"})
gdf_st.to_crs(epsg=4326).plot(ax=ax2, facecolor="none", edgecolor="black", lw=1.1, alpha=0.8)
sm  = plt.cm.ScalarMappable(cmap="RdYlGn_r", norm=mcolors.Normalize(vmin=0, vmax=99))
cb  = plt.colorbar(sm, ax=ax2, shrink=0.75, pad=0.03)
cb.set_label("Estimated Zero-Dose Rate (%) — DHIS2 Calibrated", fontsize=9)
ax2.set_title("Estimated Zero-Dose Rate by LGA\n"
              "Calibrated to Bayesian State Predictions (2026)", fontsize=11, fontweight="bold")
ax2.set_xlabel("Longitude"); ax2.set_facecolor("#E8F4F8")
fig.suptitle("Nigeria — LGA-Level Zero-Dose Spatial Analysis (774 LGAs)\n"
             "GRID3 Admin2 Boundaries | NPHCDA Digital Innovation Hub",
             fontsize=11.5, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig05_lga_gi_choropleth.png", dpi=160, bbox_inches="tight")
plt.show()


In [24]:
# Figure (detailed 2026) — Dual-panel spatial analysis for the GAVI briefing
# Left:  Gi* hotspot classification with state name + predicted ZD %.
# Right: Bayesian posterior mean predicted ZD rate (RdYlGn_r continuous).
# This is the primary "where to act" map for the NPHCDA briefing.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), facecolor=BG)

# ── Left panel: Gi* classification ────────────────────────────────────────
geo["_gi_color"] = geo["gi_class_2026"].map(HOTSPOT_COLORS).fillna("#CCC")
geo.plot(ax=ax1, color=geo["_gi_color"], edgecolor="white",
         linewidth=0.8, alpha=0.95)
gdf_lga.to_crs(epsg=4326).plot(ax=ax1, facecolor="none", edgecolor="#999",
                                linewidth=0.15, alpha=0.4)

for _, rr in geo.iterrows():
    if pd.isna(rr.get("zd_pred_2026_mean")):
        continue
    c = rr.geometry.centroid
    is_hot = "Hot" in str(rr["gi_class_2026"])
    ax1.annotate(f"{rr['state_key']}\n{rr['zd_pred_2026_mean']:.0f}%",
                 (c.x, c.y), ha="center", va="center",
                 fontsize=7, color="#111",
                 fontweight="bold" if is_hot else "normal")

ax1.set_title("Getis-Ord Gi* Hotspot Classification\n"
              "Zero-Dose Children (Predicted 2026)",
              fontsize=12, fontweight="bold", pad=10)
ax1.set_xlabel("Longitude", fontsize=10)
ax1.set_ylabel("Latitude", fontsize=10)
ax1.set_facecolor("#E8F4F8")
ax1.legend(handles=[mpatches.Patch(color=v, label=k) for k, v in HOTSPOT_COLORS.items()],
           loc="lower left", fontsize=8, title="Gi* Class", title_fontsize=9,
           framealpha=0.95, edgecolor="#CCC")

# ── Right panel: Bayesian posterior mean predicted rate ──────────────────
geo.plot(ax=ax2, column="zd_pred_2026_mean", cmap="RdYlGn_r",
         vmin=0, vmax=90, edgecolor="white", linewidth=0.8, alpha=0.95,
         missing_kwds={"color": "#DDD"})
gdf_lga.to_crs(epsg=4326).plot(ax=ax2, facecolor="none", edgecolor="#999",
                                linewidth=0.15, alpha=0.4)

for _, rr in geo.iterrows():
    if pd.isna(rr.get("zd_pred_2026_mean")):
        continue
    c = rr.geometry.centroid
    ax2.annotate(f"{rr['zd_pred_2026_mean']:.0f}%",
                 (c.x, c.y), ha="center", va="center",
                 fontsize=7, color="#111", fontweight="bold")

sm = plt.cm.ScalarMappable(cmap="RdYlGn_r",
                            norm=mcolors.Normalize(vmin=0, vmax=90))
cb = plt.colorbar(sm, ax=ax2, shrink=0.75, pad=0.03)
cb.set_label("Predicted Zero-Dose Rate (%)", fontsize=10)

ax2.set_title("Predicted Zero-Dose Rate by State (%)\n"
              "Bayesian Posterior Mean (2026)",
              fontsize=12, fontweight="bold", pad=10)
ax2.set_xlabel("Longitude", fontsize=10)
ax2.set_facecolor("#E8F4F8")

fig.suptitle("Nigeria Zero-Dose Children — Spatial Analysis 2026\n"
             "NPHCDA Digital Innovation Hub | CIDRE–Quantium Insights Consortium | "
             "GRID3 Administrative Boundaries",
             fontsize=13, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_gi_2026_detailed.png",
            dpi=160, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved: fig_gi_2026_detailed.png")


Saved: fig_gi_2026_detailed.png


In [25]:
# Figure 6 — Geopolitical Zone Summary
# Aggregates state-level estimates to Nigeria's six geopolitical zones for the
# NPHCDA executive briefing. Left panel: mean zero-dose rate by zone, observed
# (2024 NDHS) and Bayesian forecast (2026–2028). Right panel: estimated
# absolute zero-dose burden by zone in 2026 (children 12–23 months).

ZONE_ORDER = ["North West", "North East", "North Central",
              "South West", "South East", "South South"]

zone_agg = (res.groupby("zone")
               .agg(zd_2024     = ("zd_obs_2024",       "mean"),
                    zd_2026     = ("zd_pred_2026_mean", "mean"),
                    zd_2027     = ("zd_pred_2027_mean", "mean"),
                    zd_2028     = ("zd_pred_2028_mean", "mean"),
                    burden_2026 = ("zd_count_2026",     "sum"))
               .reindex(ZONE_ORDER))

# Two-line x-tick labels (e.g. "North\nWest")
xt = [z.replace(" ", "\n") for z in zone_agg.index]

# Year palette for the grouped bars on the left panel
YEAR_COLORS = {2024: "#4C72B0", 2026: "#E67E50",
               2027: "#5BA86A", 2028: "#D7574E"}

fig, (axL, axR) = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)

# ── Left panel: mean zero-dose rate by zone, grouped by year ─────────────
years = [2024, 2026, 2027, 2028]
x     = np.arange(len(zone_agg))
w     = 0.20
for i, y in enumerate(years):
    vals = zone_agg[f"zd_{y}"].values
    bars = axL.bar(x + (i - 1.5) * w, vals, w, label=str(y),
                   color=YEAR_COLORS[y], edgecolor="white", lw=0.4)
    for b, v in zip(bars, vals):
        if np.isfinite(v):
            axL.text(b.get_x() + b.get_width()/2, v + 0.7, f"{v:.1f}%",
                     ha="center", va="bottom", fontsize=8.5, color="#222")

axL.set_facecolor(BG)
axL.set_xticks(x); axL.set_xticklabels(xt, fontsize=10)
_ymax = float(np.nanmax(zone_agg[[f"zd_{y}" for y in years]].values))
axL.set_ylim(0, max(75, _ymax * 1.20))
axL.set_ylabel("Mean Zero-Dose Rate (%)", fontsize=11)
axL.set_title("Mean Zero-Dose Rate by Zone\n(Observed 2024 + Forecast 2026–2028)",
              fontsize=12, fontweight="bold", pad=10)
axL.grid(axis="y", alpha=0.2, ls="--")
axL.legend(fontsize=9, framealpha=0.9, edgecolor="#CCC")
axL.spines["top"].set_visible(False); axL.spines["right"].set_visible(False)

# ── Right panel: predicted absolute burden by zone (2026) ────────────────
burden_k      = zone_agg["burden_2026"].values / 1000.0
bar_colors_zn = [ZONE_COLORS.get(z, "#888") for z in zone_agg.index]

bars = axR.bar(x, burden_k, color=bar_colors_zn,
               edgecolor="white", lw=0.5, alpha=0.95)
_bmax = float(np.nanmax(burden_k)) if np.isfinite(np.nanmax(burden_k)) else 0
for b, v in zip(bars, burden_k):
    if np.isfinite(v):
        axR.text(b.get_x() + b.get_width()/2, v + _bmax * 0.015,
                 f"{v:.0f}k", ha="center", va="bottom",
                 fontsize=11, color="#222", fontweight="bold")

axR.set_facecolor(BG)
axR.set_xticks(x); axR.set_xticklabels(xt, fontsize=10)
axR.set_ylim(0, _bmax * 1.15 if _bmax > 0 else 1)
axR.set_ylabel("Zero-Dose Children (thousands)", fontsize=11)
axR.set_title("Predicted Zero-Dose Burden by Zone (2026)\n(Estimated children 12–23 months)",
              fontsize=12, fontweight="bold", pad=10)
axR.grid(axis="y", alpha=0.2, ls="--")
axR.spines["top"].set_visible(False); axR.spines["right"].set_visible(False)

fig.suptitle("Nigeria Zero-Dose Geopolitical Zone Summary",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig5_zone_forecast_summary.png",
            dpi=160, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved: fig5_zone_forecast_summary.png")


Saved: fig5_zone_forecast_summary.png


---
## Section 8 — Save Outputs and Final Summary

In [26]:
# Save state rankings
state_save_cols = (["state_rank","state","zone"] +
                   [f"zd_obs_{y}" for y in [2008,2013,2018,2024]] +
                   [c for yr in FORECAST_YEARS
                    for c in [f"zd_pred_{yr}_mean",f"zd_pred_{yr}_lo95",
                              f"zd_pred_{yr}_hi95",f"zd_count_{yr}"]] +
                   gi_cols + ["risk_index","priority_tier","pop_under5","cohort_12_23m"])
state_save_cols = [c for c in state_save_cols if c in res.columns]
res[state_save_cols].to_csv(f"{OUTPUT_DIR}/state_zero_dose_forecast_ranked.csv", index=False)
print(f"Saved: state_zero_dose_forecast_ranked.csv")

geo_lga[["state_key","lga_key","gi_z_lga","gi_p_lga","gi_class_lga",
          "zd_proxy_pct","lga_risk_index","lga_tier"]]    .to_csv(f"{OUTPUT_DIR}/lga_gi_star_results.csv", index=False)
print(f"Saved: lga_gi_star_results.csv")

# Summary
print(f"\n{'='*60}")
print(f"ANALYSIS COMPLETE — KEY RESULTS")
print(f"{'='*60}")
print(f"Max R-hat : {max_rh:.4f}  |  Min ESS : {min_ess:.0f}")
print(f"National predicted zero-dose rate (2026) : {res['zd_pred_2026_mean'].mean():.1f}%")
print(f"National estimated zero-dose children    : {res['zd_count_2026'].sum():,.0f}")
print(f"\nHot spots (p<0.05):")
for _,r in res[res["gi_class_2026"].str.startswith("Hot Spot",na=False)]           .sort_values("gi_z_2026",ascending=False).iterrows():
    print(f"  {r['state']:<14}  z={r['gi_z_2026']:.3f}  ZD={r['zd_pred_2026_mean']:.1f}%  [{r['gi_class_2026']}]")
print(f"\nTier 1 Critical states:")
for _,r in res[res["priority_tier"].astype(str).str.contains("Critical")].iterrows():
    print(f"  #{r['state_rank']:<2}  {r['state']:<14}  ZD={r['zd_pred_2026_mean']:.1f}%  Burden~{r['zd_count_2026']:,.0f}")
print(f"\nAll outputs in: {OUTPUT_DIR}/")


Saved: state_zero_dose_forecast_ranked.csv
Saved: lga_gi_star_results.csv

ANALYSIS COMPLETE — KEY RESULTS
Max R-hat : 1.0017  |  Min ESS : 2171
National predicted zero-dose rate (2026) : 26.2%
National estimated zero-dose children    : 2,070,369

Hot spots (p<0.05):
  Sokoto          z=2.595  ZD=71.9%  [Hot Spot (p<0.01)]
  Kebbi           z=2.220  ZD=68.3%  [Hot Spot (p<0.01)]
  Zamfara         z=1.761  ZD=73.1%  [Hot Spot (p<0.01)]
  Niger           z=1.100  ZD=45.2%  [Hot Spot (p<0.01)]
  Katsina         z=1.085  ZD=38.1%  [Hot Spot (p<0.05)]
  Kaduna          z=0.667  ZD=43.4%  [Hot Spot (p<0.05)]

Tier 1 Critical states:
  #1   Sokoto          ZD=71.9%  Burden~168,837
  #2   Kebbi           ZD=68.3%  Burden~154,105
  #3   Zamfara         ZD=73.1%  Burden~156,169

All outputs in: outputs/


---
## Section 9 — Download Results (Colab only)
Run this cell to download all output files to your local computer.

In [27]:
# ─── COLAB: Download all output files ────────────────────────────────────
# Run this cell at the end to download results to your computer.
# Each file will trigger a browser download prompt.

if IN_COLAB:
    from google.colab import files
    import os

    output_files = [
        f"outputs/{f}" for f in os.listdir("outputs")
        if f.endswith((".csv", ".png", ".ipynb"))
    ]
    output_files.sort()

    print(f"Downloading {len(output_files)} files...")
    for path in output_files:
        print(f"  ⬇  {path}")
        files.download(path)
    print("\n✅ All downloads complete.")
else:
    print("Local environment — files are already in the outputs/ folder.")
    import os
    for f in sorted(os.listdir("outputs")):
        sz = os.path.getsize(f"outputs/{f}") / 1024
        print(f"  {sz:6.0f} KB  outputs/{f}")


  ⬇  outputs/fig01_state_trajectories.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ⬇  outputs/fig02_state_forest_2026.png


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  ⬇  outputs/state_zero_dose_forecast_ranked.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All downloads complete.


In [ ]:
# ============================================================================
# [CIDRE-Quantium] CSV table-formatting pass  -  appended for GAVI review
# ----------------------------------------------------------------------------
# Purpose : make this notebook's GENERATED CSV tables presentation-ready.
#           (consistent rounding to 4 dp, integer count/rank columns, and a
#            styled .xlsx copy with a bold frozen header).
# Scope   : ONLY this notebook's own result tables, listed in _FMT_TABLES below.
#           Input/source data files are never touched.
# Safety  : runs AFTER all analysis cells; does not change any analysis or any
#           column names / row counts, so end-to-end behaviour is unchanged.
#           Every step is guarded - it can never raise and stop the notebook.
# ============================================================================
import os as _os
import numpy as _np
import pandas as _pd

try:
    _FMT_DIR = str(OUTPUT_DIR)            # this notebook's own output directory
except NameError:
    _FMT_DIR = "outputs"               # fallback if the cell is run in isolation

_FMT_TABLES = ["state_zero_dose_forecast_ranked.csv", "lga_zero_dose_ranked.csv", "lga_gi_star_results.csv", "model_diagnostics.csv"]
_INT_HINTS  = ("count", "rank", "year", "yr", "pop", "cohort",
               "n_children", "susceptible", "_n")

def _fmt_one(_path):
    if not _os.path.exists(_path):
        return False
    try:
        _df = _pd.read_csv(_path)
    except Exception as _e:
        print("[format] skip", _os.path.basename(_path), "->", _e)
        return False
    for _col in _df.select_dtypes(include=[_np.number]).columns:
        _name = str(_col).lower(); _s = _df[_col]
        if any(_h in _name for _h in _INT_HINTS) and _s.dropna().mod(1).eq(0).all():
            _df[_col] = _s.astype("Int64")          # tidy whole-number counts
        else:
            _df[_col] = _s.round(4)                 # consistent decimals
    _df.to_csv(_path, index=False)                  # same schema, tidier values
    try:                                            # styled Excel companion
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter
        _xp = _os.path.splitext(_path)[0] + ".xlsx"
        with _pd.ExcelWriter(_xp, engine="openpyxl") as _xw:
            _df.to_excel(_xw, index=False, sheet_name="Table")
            _ws = _xw.sheets["Table"]
            _fill = PatternFill("solid", fgColor="1F3B57")
            _font = Font(bold=True, color="FFFFFF")
            _thin = Side(style="thin", color="D9D9D9")
            _bd = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
            for _j, _c in enumerate(_df.columns, 1):
                _cell = _ws.cell(row=1, column=_j)
                _cell.fill = _fill; _cell.font = _font
                _cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                _ws.column_dimensions[get_column_letter(_j)].width = min(max(len(str(_c)) + 2, 10), 40)
            for _i in range(2, len(_df) + 2):
                for _j in range(1, len(_df.columns) + 1):
                    _ws.cell(row=_i, column=_j).border = _bd
            _ws.freeze_panes = "A2"; _ws.row_dimensions[1].height = 26
    except Exception as _e:
        print("[format] xlsx styling skipped for", _os.path.basename(_path), "->", _e)
    return True

_done = [_t for _t in _FMT_TABLES if _fmt_one(_os.path.join(_FMT_DIR, _t))]
if _FMT_TABLES:
    print("Table formatting complete -", len(_done), "of", len(_FMT_TABLES),
          "result table(s) formatted in '%s':" % _FMT_DIR)
    for _t in _done:
        print("   -", _t)
else:
    print("Domain produces figure/report outputs only - no result CSV tables to format.")


### 6.6 Pareto concentration of zero-dose burden across LGAs (priority view)

A stakeholder-facing prioritisation chart. LGAs are ranked by their estimated absolute zero-dose burden and plotted against the cumulative share of the national total, so the priority LGAs are visible at a glance before any table is filtered. The figure and a banded priority table are written to the output directory.

In [ ]:
# Section 6.6 (added): Pareto concentration of zero-dose burden across LGAs.
# Reads this notebook's own ranked output and writes:
#   - fig_d5_lga_pareto.png        (Pareto chart: per-LGA burden + cumulative %)
#   - D5_lga_pareto_priority.csv   (every LGA with cumulative %, priority band)
# Self-contained and guarded so it never halts an end-to-end run.
try:
    import os as _os, numpy as _np, pandas as _pd
    import matplotlib.pyplot as _plt
    from matplotlib.ticker import FuncFormatter as _FF

    _PDIR = OUTPUT_DIR if "OUTPUT_DIR" in globals() else "outputs"
    _pf = _pd.read_csv(_os.path.join(_PDIR, "lga_zero_dose_ranked.csv"))

    def _pick(cands):
        for c in cands:
            if c in _pf.columns:
                return c
        return None
    _ccount = _pick(["zd_count_est", "ZD count (est)", "zd_count"])
    _crate  = _pick(["zd_proxy_pct", "ZD proxy (%)", "zd_rate"])
    _cstate = _pick(["state", "State"]); _clga = _pick(["lga", "LGA"])
    _czone  = _pick(["zone", "Zone"]);   _ctier = _pick(["lga_tier", "LGA tier"])
    if _ccount is None:
        raise ValueError("zero-dose count column not found in ranked file")

    _d = _pf.dropna(subset=[_ccount]).sort_values(_ccount, ascending=False).reset_index(drop=True)
    _n = len(_d); _tot = float(_d[_ccount].sum())
    _d["_rank"] = _np.arange(1, _n + 1)
    _d["_cum"]  = 100.0 * _d[_ccount].cumsum() / _tot
    _t20 = int(_np.ceil(0.20 * _n)); _s20 = float(_d.loc[_t20 - 1, "_cum"])
    _n80 = int((_d["_cum"] >= 80).idxmax()) + 1; _p80 = 100.0 * _n80 / _n

    NAVY = "#1F3B57"; ACCENT = "#C0392B"; GREYB = "#C3CED6"; MUTE = "#5B6B79"
    fig, ax1 = _plt.subplots(figsize=(12, 6.2), facecolor="white")
    ax1.bar(_d["_rank"], _d[_ccount], width=1.0,
            color=[ACCENT if i < _t20 else GREYB for i in range(_n)], linewidth=0)
    ax1.set_xlabel("LGAs ranked by estimated zero-dose burden (highest to lowest)", fontsize=11)
    ax1.set_ylabel("Estimated zero-dose children per LGA", fontsize=11, color=MUTE)
    ax1.set_xlim(0, _n + 1); ax1.set_ylim(0, _d[_ccount].max() * 1.08)
    ax1.tick_params(axis="y", labelcolor=MUTE)
    ax2 = ax1.twinx(); ax2.plot(_d["_rank"], _d["_cum"], color=NAVY, lw=2.4)
    ax2.set_ylabel("Cumulative % of national zero-dose burden", fontsize=11, color=NAVY)
    ax2.set_ylim(0, 101); ax2.tick_params(axis="y", labelcolor=NAVY)
    ax2.yaxis.set_major_formatter(_FF(lambda v, _: f"{v:.0f}%"))
    ax2.axvline(_t20, color=ACCENT, ls="--", lw=1.3, alpha=0.8)
    ax2.scatter([_t20], [_s20], s=55, color=ACCENT, zorder=6, edgecolor="white", lw=1)
    ax2.annotate(f"Top 20% of LGAs ({_t20})\ncarry {_s20:.0f}% of the burden",
                 xy=(_t20, _s20), xytext=(_t20 + max(40, _n * 0.09), _s20 - 20),
                 fontsize=9.5, color="#7F1D1D", fontweight="bold",
                 bbox=dict(facecolor="#FEF2F2", edgecolor=ACCENT, lw=0.8, pad=4, alpha=0.95),
                 arrowprops=dict(arrowstyle="->", color=ACCENT, lw=1))
    ax2.axhline(80, color=MUTE, ls=":", lw=1.1, alpha=0.7)
    ax2.axvline(_n80, color=NAVY, ls=":", lw=1.1, alpha=0.6)
    ax2.scatter([_n80], [80], s=55, color=NAVY, zorder=6, edgecolor="white", lw=1)
    ax2.annotate(f"80% of burden sits in the\ntop {_n80} LGAs ({_p80:.0f}% of all LGAs)",
                 xy=(_n80, 80), xytext=(_n80 + 20, 52),
                 fontsize=9.5, color=NAVY, fontweight="bold",
                 bbox=dict(facecolor="#EAF1F5", edgecolor=NAVY, lw=0.8, pad=4, alpha=0.95),
                 arrowprops=dict(arrowstyle="->", color=NAVY, lw=1))
    ax1.axvspan(0, _t20, color=ACCENT, alpha=0.05)
    ax1.set_title("Domain 5 - Pareto concentration of zero-dose burden across LGAs\n"
                  f"{_tot:,.0f} estimated zero-dose children across {_n} LGAs",
                  fontsize=12.5, fontweight="bold", color=NAVY, loc="left", pad=12)
    ax1.spines["top"].set_visible(False); ax2.spines["top"].set_visible(False)
    _plt.tight_layout()
    _figp = _os.path.join(_PDIR, "fig_d5_lga_pareto.png")
    _plt.savefig(_figp, dpi=150, bbox_inches="tight", facecolor="white")
    _plt.show()

    def _band(c):
        return "A: drives first 50%" if c <= 50 else ("B: 50-80%" if c <= 80 else "C: long tail")
    _keep = [c for c in [_cstate, _clga, _czone, _ccount, _crate, _ctier] if c]
    _out = _d[_keep].copy()
    _out.insert(0, "burden_rank", _d["_rank"])
    _out["cum_pct_of_burden"] = _d["_cum"].round(1)
    _out["priority_band"] = _d["_cum"].apply(_band)
    _out["top_20pct_lga"] = _np.where(_d["_rank"] <= _t20, "Yes", "No")
    _csvp = _os.path.join(_PDIR, "D5_lga_pareto_priority.csv")
    _out.to_csv(_csvp, index=False)
    print(f"[Pareto] figure -> {_figp}")
    print(f"[Pareto] priority table -> {_csvp}")
    print(f"[Pareto] {_n} LGAs, {_tot:,.0f} ZD children | "
          f"top 20% ({_t20}) = {_s20:.1f}% | 80% of burden = top {_n80} LGAs ({_p80:.1f}%)")
except Exception as _e:
    print("[Pareto] skipped:", _e)


In [ ]:
# =====================================================================
# APPENDED (additive): accurate, reproducible LGA burden ranking + export.
# The earlier cell ranks LGAs by a within-state-normalised risk index
# (national_lga_rank = lga_risk_index.rank(method="min")), which is not
# comparable across states and produces tied/gapped national ranks. This
# cell re-derives a valid NATIONAL rank by zero-dose burden, a contiguous
# within-state rank, cleans LGA names, and validates completeness before
# writing the analysis-ready CLEAN file. Existing cells are unchanged.
# =====================================================================
import pandas as _pd, numpy as _np, os as _os, re as _re
_src = _os.path.join(OUTPUT_DIR, "lga_zero_dose_ranked.csv")
_r = _pd.read_csv(_src)
_ren = {"zone":"Zone","state":"State","lga":"LGA","penta_1_count":"Penta1 count",
        "penta_3_count":"Penta3 count","p1_share":"Penta1 share","zd_proxy_pct":"ZD proxy (%)",
        "zd_count_est":"ZD count (est)","dropout_p1p3":"Dropout P1-P3 (%)",
        "lga_risk_index":"LGA risk index","lga_tier":"LGA tier"}
_r = _r.rename(columns={k:v for k,v in _ren.items() if k in _r.columns})

# Clean LGA names: drop any leading 2-letter state code and the trailing descriptor
def _clean_name(x):
    s = str(x).strip()
    s = _re.sub(r"^[a-z]{2}\s+", "", s, flags=_re.IGNORECASE)
    s = _re.sub(r"\s*local\s+government\s+area\s*$", "", s, flags=_re.IGNORECASE)
    return s.strip()
_r["LGA"] = _r["LGA"].map(_clean_name)

# Completeness guard: surface upstream fragility instead of shipping a near-empty file
_est = _r[_r["ZD count (est)"].notna()].copy()
if len(_est) < 600:
    print(f"  WARNING: only {len(_est)} LGAs have a burden estimate (expected ~730). "
          f"Verify the population-denominator merge in the LGA generation cell before using this output.")

# Valid NATIONAL rank by burden (comparable across states), contiguous + deterministic
_est = _est.sort_values("ZD count (est)", ascending=False, kind="stable").reset_index(drop=True)
_est["National rank"] = _np.arange(1, len(_est) + 1)
_est["State rank"] = _est.groupby("State")["ZD count (est)"].rank(ascending=False, method="first").astype(int)

_cols = ["National rank","State rank","Zone","State","LGA","Penta1 count","Penta3 count",
         "Penta1 share","ZD proxy (%)","ZD count (est)","Dropout P1-P3 (%)","LGA risk index","LGA tier"]
_clean = _est[[c for c in _cols if c in _est.columns]]
assert sorted(_clean["National rank"].tolist()) == list(range(1, len(_clean) + 1)), "national rank not contiguous"
_clean.to_csv(_os.path.join(OUTPUT_DIR, "D5_lga_zero_dose_ranked_CLEAN.csv"), index=False)
print(f"  Saved D5_lga_zero_dose_ranked_CLEAN.csv: {len(_clean)} estimable LGAs, "
      f"National rank 1..{len(_clean)} contiguous; total estimated zero-dose children "
      f"{_clean['ZD count (est)'].sum():,.0f}.")


### 6.7 Population-weighted LGA burden (administrative LGA population dataset)

The even split in the LGA generation cell gives every LGA in a state the same cohort, so two LGAs at the same modelled rate get identical counts. This step replaces that with a population weighting: each state's survey-anchored 12-to-23-month cohort is distributed across its LGAs in proportion to LGA population (NPC 2022 projection, the `administrative_lga_population.csv` dataset). Counts then vary by LGA (`zero-dose children = rate x population share`), while the rates and every state, zone and national total stay exactly as before. The matcher reconciles LGA-name spelling variants and reassigns the six FCT LGAs that the source file mislabels under Enugu. This is the Domain 5 step that consumes the LGA population dataset.

In [ ]:
# =====================================================================
# APPENDED (additive): population-weighted LGA zero-dose burden.
# Distributes each state's survey-anchored 12-23m cohort across its LGAs
# in proportion to LGA population (NPC 2022 projection, the administrative
# LGA population dataset) instead of an equal split, so absolute counts
# vary by LGA while every state, zone and national total is preserved.
# Rates are unchanged. Rewrites CLEAN, Pareto and priority tables and the
# Pareto figure. Earlier cells are unchanged.
#
# SOURCE of administrative_lga_population.csv: City Population (citypopulation.de),
#   "Nigeria: Administrative Division (States and Local Government Areas)",
#   https://www.citypopulation.de/en/nigeria/admin/  -- compiled from the National
#   Population Commission of Nigeria (2006 Census) and National Bureau of Statistics.
#   The 2022 column is a projection using a uniform within-state growth rate, so
#   within-state population shares follow the 2006 census proportions. Official NPC
#   or GRID3 LGA figures can replace this file with no change to the method.
# =====================================================================
import pandas as _pd, numpy as _np, os as _os, re as _re, difflib as _dl

_cands = ["administrative_lga_population.csv",
          _os.path.join(OUTPUT_DIR, "administrative_lga_population.csv"),
          "data/administrative_lga_population.csv",
          "/mnt/user-data/uploads/administrative_lga_population.csv"]
_poppath = next((p for p in _cands if _os.path.exists(p)), None)
assert _poppath, "administrative_lga_population.csv not found; place it beside this notebook."
_pop = _pd.read_csv(_poppath)
_pop = _pop[_pop["Status"].str.strip() == "Local Government Area"].copy()

# source quirk: 6 FCT LGAs are mislabelled under 'Enugu' -> reassign to FCT
_FCT6 = {"Abaji","Abuja Municipal Area Council","Bwari","Gwagwalada","Kuje","Kwali"}
_pop.loc[_pop["Name"].isin(_FCT6), "State"] = "FCT"

def _nstate(s):
    s = str(s).lower().strip()
    return "fct" if ("federal capital" in s or s == "fct") else _re.sub(r"\(.*?\)", "", s).strip()
def _nlga(s):
    s = _re.sub(r"\(.*?\)|\[.*?\]", "", str(s).lower().strip())
    return _re.sub(r"\s+", " ", s.replace("'","").replace("/"," ").replace("-"," ").replace(".","")).strip()
def _tok(s): return " ".join(sorted(_nlga(s).split()))
_ALIAS = {("kebbi","arewa"):"arewa dandi", ("imo","ezinihitte mbaise"):"ezinihitte"}
_pop["ns"]=_pop["State"].map(_nstate); _pop["nl"]=_pop["Name"].map(_nlga); _pop["tk"]=_pop["Name"].map(_tok)
_pop["P"]=_pd.to_numeric(_pop["PopulationProjection2022-03-21"], errors="coerce")

def _match(st, nl, tk):
    nl = _ALIAS.get((st, nl), nl)
    sub = _pop[_pop["ns"] == st]
    for col, val in [("nl", nl), ("tk", tk)]:
        m = sub[sub[col] == val]
        if len(m): return float(m["P"].iloc[0])
    c = _dl.get_close_matches(nl, list(sub["nl"]), n=1, cutoff=0.80)
    return float(sub[sub["nl"]==c[0]]["P"].iloc[0]) if c else _np.nan

_cl = _pd.read_csv(_os.path.join(OUTPUT_DIR, "D5_lga_zero_dose_ranked_CLEAN.csv"))
_cl.columns = [c.strip() for c in _cl.columns]
_cl["LGA population (2022)"] = [_match(_nstate(s), _nlga(l), _tok(l)) for s,l in zip(_cl["State"], _cl["LGA"])]
_miss = int(_cl["LGA population (2022)"].isna().sum())
print(f"  LGA population matched: {len(_cl)-_miss}/{len(_cl)} ({100*(len(_cl)-_miss)/len(_cl):.1f}%)")
_cl["LGA population (2022)"] = _cl.groupby("State")["LGA population (2022)"].transform(
    lambda s: s.fillna(s.mean() if s.notna().any() else 1.0))

_rate = _pd.to_numeric(_cl["ZD proxy (%)"], errors="coerce")/100.0
_cur  = _pd.to_numeric(_cl["ZD count (est)"], errors="coerce")     # even-split burden from prior cell
_raw  = _rate * _cl["LGA population (2022)"]
_scur = _cur.groupby(_cl["State"]).transform("sum")               # preserve each state's total
_sraw = _raw.groupby(_cl["State"]).transform("sum")
_cl["ZD count (est)"] = (_raw * _scur / _sraw).round(0).astype(int)
_cl["LGA population (2022)"] = _cl["LGA population (2022)"].round(0).astype(int)

_sevmap = {"Tier 1":"Critical","Tier 2":"High","Tier 3":"Moderate","Tier 4":"Lower"}
_cl["Severity (within state)"] = _cl["LGA tier"].astype(str).str.strip().map(_sevmap)

_cl = _cl.sort_values("ZD count (est)", ascending=False, kind="stable").reset_index(drop=True)
_cl["National rank"] = _np.arange(1, len(_cl)+1)
_cl["State rank"] = _cl.groupby("State")["ZD count (est)"].rank(ascending=False, method="first").astype(int)
_order = ["National rank","State rank","Zone","State","LGA","Penta1 count","Penta3 count","Penta1 share",
          "ZD proxy (%)","LGA population (2022)","ZD count (est)","Dropout P1-P3 (%)","LGA risk index",
          "LGA tier","Severity (within state)"]
_cl = _cl[[c for c in _order if c in _cl.columns]]
_cl.to_csv(_os.path.join(OUTPUT_DIR,"D5_lga_zero_dose_ranked_CLEAN.csv"), index=False)
try: _cl.to_excel(_os.path.join(OUTPUT_DIR,"D5_lga_zero_dose_ranked_CLEAN.xlsx"), index=False)
except Exception as _e: print("   (xlsx skipped:", _e, ")")

_tot = _cl["ZD count (est)"].sum()
_par = _cl.sort_values("ZD count (est)", ascending=False).reset_index(drop=True)
_out = _pd.DataFrame({
    "Burden rank": _np.arange(1, len(_par)+1), "State": _par["State"], "LGA": _par["LGA"], "Zone": _par["Zone"],
    "LGA population (2022)": _par["LGA population (2022)"], "Zero-dose children (est)": _par["ZD count (est)"],
    "Zero-dose rate (%)": _pd.to_numeric(_par["ZD proxy (%)"]).round(2), "LGA tier": _par["LGA tier"],
    "Severity (within state)": _par["Severity (within state)"]})
_cum = _out["Zero-dose children (est)"].cumsum()/_tot*100
_out["Cumulative % of burden"] = _cum.round(1)
_out["Priority band"] = _np.where(_cum<=50,"A: drives first 50%",_np.where(_cum<=80,"B: 50-80%","C: long tail 80-100%"))
_n20 = int(round(0.20*len(_out))); _out["Top 20% LGA"] = _np.where(_out["Burden rank"]<=_n20,"Yes","No")
_out.to_csv(_os.path.join(OUTPUT_DIR,"D5_lga_pareto_priority.csv"), index=False)
try: _out.to_excel(_os.path.join(OUTPUT_DIR,"D5_lga_pareto_priority.xlsx"), index=False)
except Exception: pass
_out.head(60).to_csv(_os.path.join(OUTPUT_DIR,"D5_top60_priority_LGAs.csv"), index=False)
_out.head(20).to_csv(_os.path.join(OUTPUT_DIR,"D5_top20_priority_LGAs.csv"), index=False)

# regenerate the Pareto figure from the population-weighted distribution
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as _plt
from matplotlib.ticker import FuncFormatter as _FF
_NAVY="#1F3B57"; _STEEL="#2E6E8E"; _ACC="#C0392B"; _MUTE="#6B7A88"
_b=_out["Zero-dose children (est)"].values; _x=_np.arange(1,len(_b)+1)
_t20=int(round(0.20*len(_b))); _t80=int((_cum<=80).sum()+1)
_fig,_ax=_plt.subplots(figsize=(11.0,4.99),dpi=192)
_ax.bar(_x,_b,width=1.0,color=_STEEL,linewidth=0); _ax.bar(_x[:_t20],_b[:_t20],width=1.0,color=_ACC,linewidth=0)
_ax.set_ylabel("Estimated zero-dose children per LGA",fontsize=10,color=_NAVY)
_ax.set_xlabel("LGAs ranked by estimated zero-dose burden (highest to lowest)",fontsize=10,color=_NAVY)
_ax.set_xlim(0,len(_b)+1); _ax.yaxis.set_major_formatter(_FF(lambda v,_:f"{int(v/1000)}k" if v>=1000 else f"{int(v)}"))
for _sp in ["top","right"]: _ax.spines[_sp].set_visible(False)
_ax2=_ax.twinx(); _ax2.plot(_x,_cum.values,color=_NAVY,lw=2.0); _ax2.set_ylim(0,100)
_ax2.set_ylabel("Cumulative % of national burden",fontsize=10,color=_NAVY); _ax2.spines["top"].set_visible(False)
_ax2.yaxis.set_major_formatter(_FF(lambda v,_:f"{int(v)}%"))
_ax2.axvline(_t20,color=_ACC,ls=":",lw=1.2); _ax2.axhline(80,color=_MUTE,ls=":",lw=1.0); _ax2.axvline(_t80,color=_MUTE,ls=":",lw=1.0)
_ax2.annotate(f"Top 20% ({_t20} LGAs) = {_cum.iloc[_t20-1]:.0f}% of burden",xy=(_t20,_cum.iloc[_t20-1]),
              xytext=(_t20+30,46),fontsize=9,color=_ACC,fontweight="bold",arrowprops=dict(arrowstyle="-",color=_ACC,lw=1))
_ax.set_title(f"All {len(_b)} reporting LGAs ranked by estimated zero-dose children, with cumulative share of the {_tot/1e6:.2f} million national total",
              fontsize=11,color=_NAVY,fontweight="bold",pad=10,loc="left")
_fig.tight_layout(); _fig.savefig(_os.path.join(OUTPUT_DIR,"fig_d5_lga_pareto.png"),dpi=192,bbox_inches="tight",facecolor="white"); _plt.close(_fig)

print(f"  Population-weighted burden written. National total {_tot:,.0f} (preserved); "
      f"distinct counts {_cl['ZD count (est)'].nunique()}/{len(_cl)}; "
      f"top 20% = {_cum.iloc[_n20-1]:.0f}% of burden; 80% reached at {_t80} LGAs.")
